4) CSV 로더와 데이터프레임 로더

In [ ]:
# [목적] CSV 파일을 LangChain Document 목록으로 읽어 기본 구조를 확인하는 예제
# CSVLoader는 표의 각 행을 검색 가능한 Document로 변환하며, load()가 변환 결과를 docs에 저장합니다.
# 문서 수와 첫 문서의 메타데이터를 확인해 CSV가 올바르게 읽혔는지 점검합니다.
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path = "../data/titanic.csv"
)  # CSV 로더 생성

docs = loader.load()  # 데이터 로드

print(len(docs))
print(docs[0].metadata)

In [ ]:
# [목적] CSV에서 변환된 한 행의 본문 내용을 확인하는 예제
# docs[1]은 두 번째 행을 뜻하며, page_content에는 열 이름과 값이 텍스트로 정리되어 있습니다.
# 검색에 사용할 문서 본문이 어떤 형태로 만들어지는지 확인합니다.
print(docs[1].page_content)

In [ ]:
# [목적] CSV의 구분 규칙과 열 이름을 직접 지정해 로더를 설정하는 예제
# csv_args는 쉼표 구분, 따옴표 처리, 열 이름처럼 CSV를 해석하는 규칙을 전달하는 설정 딕셔너리입니다.
# 헤더가 없거나 열 구성이 불명확한 CSV도 일정한 구조의 Document로 변환하기 위해 사용합니다.
loader = CSVLoader(
    file_path = "../data/titanic.csv",
    csv_args={
        "delimiter": ",",  # 구분자
        "quotechar": '"',  # 인용 부호 문자
        "fieldnames": [
            "Passenger ID",
            "Survival (1: Survived, 0: Died)",
            "Passenger Class",
            "Name",
            "Sex",
            "Age",
            "Number of Siblings/Spouses Aboard",
            "Number of Parents/Children Aboard",
            "Ticket Number",
            "Fare",
            "Cabin",
            "Port of Embarkation",
        ],  # 필드 이름
    },
)

In [ ]:
# [목적] 사용자 설정으로 CSV를 읽은 결과를 확인하는 예제
# 앞 셀에서 설정한 loader로 행들을 Document 목록으로 변환한 뒤 두 번째 행의 본문을 출력합니다.
# 지정한 열 이름과 구분 규칙이 실제 결과에 제대로 반영됐는지 검토합니다.
docs = loader.load()  # 데이터 로드

print(docs[1].page_content)  # 데이터 출력

In [ ]:
# [목적] 한 CSV 행을 XML과 비슷한 태그 구조의 문자열로 바꾸는 예제
# 행의 각 '열 이름: 값' 줄을 나누고, 열 이름을 태그로 사용해 값이 구분된 row_str을 만듭니다.
# 구조화된 텍스트는 LLM이 각 값의 의미를 더 명확히 구분하도록 전달할 때 활용할 수 있습니다.
row = docs[1].page_content.split("\n")
row_str = "<row>"

for element in row:
    splitted_element = element.split(":")
    value = splitted_element[-1]
    col = ":".join(splitted_element[:-1])
    row_str += f"<{col}>{value.strip()}</{col}>"

row_str += "</row>"

print(row_str)

In [ ]:
# [목적] 모든 CSV 행을 태그 구조의 문자열로 순차 변환하는 예제
# docs의 각 Document에 대해 앞 셀과 같은 변환 과정을 반복해 행별 row 태그 문자열을 생성합니다.
# 표 전체를 구조화된 텍스트로 바꿔 저장하거나 모델 입력으로 가공하는 흐름의 예시입니다.
for doc in docs:
    row = doc.page_content.split("\n")
    row_str = "<row>"

    for element in row:
        splitted_element = element.split(":")
        value = splitted_element[-1]
        col = ":".join(splitted_element[:-1])
        row_str += f"<{col}>{value.strip()}</{col}>"

    row_str += "</row>"
    print(row_str)

In [ ]:
# [목적] 특정 CSV 열을 각 Document의 출처 정보로 지정하는 예제
# source_column은 행을 구분할 열을 metadata의 source 값으로 사용하도록 CSVLoader에 알려 줍니다.
# PassengerId처럼 고유한 값을 출처로 두면 검색 결과가 어느 행에서 왔는지 추적하기 쉽습니다.
loader = CSVLoader(
    file_path = "../data/titanic.csv",
    source_column="PassengerId"
)  # CSV 로더 설정, 파일 경로 및 소스 컬럼 지정

docs = loader.load()  # 데이터 로드

print(docs[1])  # 데이터 출력

UnstructuredCSVLoader

In [ ]:
# [목적] UnstructuredCSVLoader로 CSV를 요소 단위 문서로 읽는 예제
# UnstructuredCSVLoader는 표 구조를 분석해 문서를 만들며, mode='elements'는 분석된 요소 단위 결과를 반환합니다.
# metadata의 text_as_html을 확인해 표가 HTML 형태로도 보존됐는지 살펴봅니다.
from langchain_community.document_loaders.csv_loader import UnstructuredCSVLoader

loader = UnstructuredCSVLoader(
    file_path = "../data/titanic.csv",
    mode="elements"
)

docs = loader.load()  # 문서 로드

print(docs[0].metadata["text_as_html"][:1000])

DataFrameLoader

In [ ]:
# [목적] pandas로 CSV를 표 형태의 DataFrame으로 읽는 예제
# DataFrame은 행과 열을 가진 표 데이터 객체로, CSV 내용을 확인하거나 원하는 열을 선택하기에 편리합니다.
# 다음 셀에서 내용을 점검한 뒤 DataFrameLoader의 입력으로 사용합니다.
import pandas as pd

df = pd.read_csv("../data/titanic.csv")

In [ ]:
# [목적] DataFrame에 읽힌 CSV 데이터의 앞부분을 미리 보는 예제
# head()는 기본적으로 처음 다섯 행을 표시하므로 열 이름과 값의 형태를 빠르게 확인할 수 있습니다.
# 어떤 열을 문서 본문으로 사용할지 결정하기 전에 데이터 구조를 점검합니다.
df.head()

In [ ]:
# [목적] DataFrame의 특정 열을 중심으로 LangChain Document를 만드는 예제
# DataFrameLoader는 표의 각 행을 Document로 바꾸고, page_content_column은 본문으로 사용할 열을 지정합니다.
# 여기서는 Name을 본문으로 두고 나머지 열은 메타데이터로 보관해 검색과 결과 설명에 활용합니다.
from langchain_community.document_loaders import DataFrameLoader

loader = DataFrameLoader(
    df,
    page_content_column="Name"
)

docs = loader.load()  # 문서 로드

print(docs[0].page_content)  # 데이터 출력

print(docs[0].metadata)  # 메타데이터 출력

지연 로드

In [ ]:
# [목적] DataFrame 문서를 지연 로딩 방식으로 한 행씩 처리하는 예제
# lazy_load()는 모든 행을 한꺼번에 목록으로 만들지 않고 필요할 때 Document를 하나씩 반환합니다.
# 큰 표 데이터를 처리할 때 메모리 사용을 줄이면서 각 행의 변환 결과를 확인할 수 있습니다.
for row in loader.lazy_load():
    print(row)